In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import optuna

from copy import deepcopy
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    classification_report,
)

dataset = pd.read_csv("../../data/processing/processing.csv", index_col=0)

X = dataset.drop(columns=["Attrition"])
y = dataset["Attrition"]

scaler = StandardScaler()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train_scaler = scaler.fit_transform(X_train)
X_test_scaler = scaler.transform(X_test)

X_train = torch.tensor(X_train_scaler, dtype=torch.float32)
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.float32).reshape(-1, 1)

X_test = torch.tensor(X_test_scaler.to_numpy(), dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


def build_model(trial):
    n_layers = trial.suggest_int("n_layers", 1, 3)
    layers = []

    in_features = 41
    for i in range(n_layers):
        out_features = trial.suggest_int(f"n_units_l{i}", 16, 128)
        layers.append(nn.Linear(in_features, out_features))
        layers.append(nn.BatchNorm1d(out_features))
        layers.append(nn.ReLU())

        # 드롭아웃 비율을 0.1에서 0.5 사이로 탐색
        dropout_rate = trial.suggest_float(f"dropout_l{i}", 0.1, 0.5)
        layers.append(nn.Dropout(dropout_rate))

        in_features = out_features

    layers.append(nn.Linear(in_features, 1))

    return nn.Sequential(*layers)


def objective(trial):
    model = build_model(trial)

    # 학습률을 0.0001에서 0.01 사이(로그 스케일)로 탐색
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    epochs = 30

    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                output = model(X_batch)
                val_loss += criterion(output, y_batch).item()

        val_loss /= len(test_loader)

        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return val_loss


print("Optuna 최적화를 시작합니다...")
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

print("\n--- Optuna 탐색 완료 ---")
print("최적의 하이퍼파라미터: ", study.best_params)
print("최저 Validation Loss: ", study.best_value)
print("-" * 50)


best_params = study.best_params


class BestClassifier(nn.Module):
    def __init__(self, best_params):
        super().__init__()
        layers = []
        in_features = 41
        n_layers = best_params["n_layers"]

        for i in range(n_layers):
            out_features = best_params[f"n_units_l{i}"]
            layers.append(nn.Linear(in_features, out_features))
            layers.append(nn.BatchNorm1d(out_features))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(best_params[f"dropout_l{i}"]))
            in_features = out_features

        layers.append(nn.Linear(in_features, 1))
        self.feature_extractor = nn.Sequential(*layers)

    def forward(self, x):
        return self.feature_extractor(x)


final_model = BestClassifier(best_params)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(final_model.parameters(), lr=best_params["lr"])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

epochs = 100
best_val_loss = float("inf")
patience = 15
patience_counter = 0
best_model_weights = None

print("최적화된 아키텍처로 최종 학습을 시작합니다...")
for epoch in range(epochs):
    final_model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = final_model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    final_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            output = final_model(X_batch)
            loss = criterion(output, y_batch)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(test_loader)
    scheduler.step(avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_model_weights = deepcopy(final_model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"Early stopping triggered at epoch {epoch + 1}!")
        break

if best_model_weights is not None:
    final_model.load_state_dict(best_model_weights)

final_model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = final_model(X_batch)
        probs = torch.sigmoid(output)
        preds = (probs >= 0.5).float()

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y_batch.cpu().numpy())

print("\n--- 최종 최고 성능 모델 테스트 평가 결과 ---")
print(f"Accuracy  : {accuracy_score(all_targets, all_preds):.4f}")
print(f"Recall    : {recall_score(all_targets, all_preds):.4f}")
print(f"Precision : {precision_score(all_targets, all_preds):.4f}")
print(f"F1 Score  : {f1_score(all_targets, all_preds):.4f}")
print("\n[ Classification Report ]")
print(classification_report(all_targets, all_preds))

[I 2026-08-24 15:39:42,445] A new study created in memory with name: no-name-c20fe830-f5f6-4e6c-8179-345ccd0b0e7b


Optuna 최적화를 시작합니다...


[I 2026-08-24 15:41:34,949] Trial 0 finished with value: 0.4825168758790123 and parameters: {'n_layers': 2, 'n_units_l0': 99, 'dropout_l0': 0.1799192555416561, 'n_units_l1': 94, 'dropout_l1': 0.40276728150318203, 'lr': 0.0004138891901992785}. Best is trial 0 with value: 0.4825168758790123.
[I 2026-08-24 15:42:47,601] Trial 1 finished with value: 0.4855283774475363 and parameters: {'n_layers': 1, 'n_units_l0': 27, 'dropout_l0': 0.42434975822837756, 'lr': 0.004576671719832699}. Best is trial 0 with value: 0.4825168758790123.
[I 2026-08-24 15:44:41,043] Trial 2 finished with value: 0.4825805110089919 and parameters: {'n_layers': 1, 'n_units_l0': 80, 'dropout_l0': 0.3281218550572178, 'lr': 0.0010036257155428402}. Best is trial 0 with value: 0.4825168758790123.
[I 2026-08-24 15:47:32,307] Trial 3 finished with value: 0.4815776889974421 and parameters: {'n_layers': 3, 'n_units_l0': 49, 'dropout_l0': 0.48598045676640056, 'n_units_l1': 89, 'dropout_l1': 0.49332657130054014, 'n_units_l2': 121, 


--- Optuna 탐색 완료 ---
최적의 하이퍼파라미터:  {'n_layers': 3, 'n_units_l0': 69, 'dropout_l0': 0.27923302930408395, 'n_units_l1': 36, 'dropout_l1': 0.23758948376684902, 'n_units_l2': 79, 'dropout_l2': 0.25815699387613034, 'lr': 0.002856942975419598}
최저 Validation Loss:  0.4801659807164401
--------------------------------------------------
최적화된 아키텍처로 최종 학습을 시작합니다...
Early stopping triggered at epoch 38!

--- 최종 최고 성능 모델 테스트 평가 결과 ---
Accuracy  : 0.7536
Recall    : 0.7682
Precision : 0.7635
F1 Score  : 0.7658

[ Classification Report ]
              precision    recall  f1-score   support

         0.0       0.74      0.74      0.74      5668
         1.0       0.76      0.77      0.77      6252

    accuracy                           0.75     11920
   macro avg       0.75      0.75      0.75     11920
weighted avg       0.75      0.75      0.75     11920



In [10]:
import joblib
import os

# =====================================================
# 모델 저장 경로
# =====================================================

model_dir = "../../src/models"

os.makedirs(model_dir, exist_ok=True)

model_path = os.path.join(model_dir, "mlp.joblib")


# =====================================================
# 원본 Feature
# =====================================================

target_column = "Attrition"

raw_feature_columns = [
    column for column in dataset.columns if column not in ["Employee ID", target_column]
]


# =====================================================
# 모델 Artifact
# =====================================================

artifact = {
    # PyTorch 모델 가중치
    "model_state_dict": final_model.state_dict(),
    # Optuna 최적 파라미터
    "best_params": best_params,
    # 모델에 실제 들어가는 41개 Feature 순서
    "feature_columns": X.columns.tolist(),
    # 사용자가 입력하는 원본 Feature
    "raw_feature_columns": raw_feature_columns,
    # 원본 데이터
    "raw_data": dataset[raw_feature_columns].copy(),
}


# =====================================================
# Joblib 저장
# =====================================================

joblib.dump(artifact, model_path)

print("모델 저장 완료")
print(f"저장 위치: {model_path}")

모델 저장 완료
저장 위치: ../../src/models/mlp.joblib
